# Q.ANT's NPU: Function Learning with Fourier Layers

## Fourier Layers for Function Learning

This notebook demonstrates **Fourier layers** running on Q.ANT's Native Processing Unit (NPU) for learning 1D functions. Their performance is compared against traditional Multi-Layer Perceptrons (MLPs), which are used as essential building block in most neural networks. \
Note, the 1D function is used as a placeholder to estimate the learning capability of the respective model. Large advantages are found where the application matches the underlying Fourier basis functions which naturally capture complex, high-frequency patterns.

### What are Fourier Layers?

A Fourier layer computes a learnable Fourier series for each connection between input and output neurons:

$$f(x) = \sum_{i,k} a_{oik} \cos(k x_i + \varphi_{oik})$$

where $o, i, k$ are the indices for the output, input and the grid. The amplitudes $a_{oik}$ and phases $\varphi_{oik}$ are learned during training. Stacking multiple such layers yields a flexible neural network that excels at learning periodic and high-frequency patterns.

**Key advantages:**
- **Superior high-frequency learning**: Fourier layers capture complex, high-frequency patterns that MLPs often struggle with
- **Parameter efficiency**: Often achieve better performance with fewer trainable parameters
- **NPU-native**: Cosine evaluation maps directly to Q.ANT's photonic NPU hardware
- **Neural Operators**: Learning the solution operator of PDEs with potentially high generalization.

### Why Fourier Layers Excel on NPU Hardware

Q.ANT's Native Processing Unit (NPU) evaluates nonlinear functions such as cosine directly in the optical domain using photonic components — no iterative numerical approximation needed. This makes Fourier layers a natural fit for the hardware.

# Let's start coding

#### Import required libraries and modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qant_layers  # Have a short look at this file to see how the layers are implemented
import torch

import qant_native_computing_toolkit as qant

torch.manual_seed(10)
rng = np.random.default_rng(seed=20)

#### 1D Function To Learn

The target function combines two trigonometric terms at different frequencies:

$$f(x) = \frac{1}{2}\cos(2\pi x) + \frac{1}{4}\sin(8\pi x)$$

The function will serve as a simple benchmark for comparing the learning ability of both models.

In [ ]:
def func_1d(x: np.ndarray) -> np.ndarray:
    return 0.5 * np.cos(2 * np.pi * x) + 0.25 * np.sin(2 * np.pi * 4 * x)

#### Visualize the Function to Be Learned

Plot the ground truth over $x \in [-2, 2]$ before training either model. Notice the two oscillations with different frequencies.

In [ ]:
# Define the range of x values for learning the function
min_x = -2.0
max_x = 2.0
x = np.linspace(min_x, max_x, 1000)

plt.figure(figsize=(8, 4))
plt.plot(x, func_1d(x), label="Ground Truth Function")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## Conventional MLP

A Multi-Layer Perceptron (MLP) uses stacked `QLinear` layers with ReLU activations. `QLinear` is Q.ANT's drop-in replacement for `torch.nn.Linear`: it trains on the CPU using standard PyTorch autograd, and automatically routes computation through the Q.ANT SDK (either cpu-backend or hardware) when the model is placed in `eval()` mode.

#### Initialization

The architecture used here has three layers using ReLU with widths `[1, 32, 32, 1]`, giving a moderate parameter budget for this simple 1D problem.

In [ ]:
class ConventionalMLP(torch.nn.Module):
    """MLP using QLinear layers — training runs on CPU, inference runs on NPU (eval mode)."""

    def __init__(self, layer_sizes, activation=torch.relu):
        super().__init__()
        self.layers = torch.nn.ModuleList(
            [
                qant_layers.QLinear(layer_sizes[i], layer_sizes[i + 1])
                for i in range(len(layer_sizes) - 1)
            ]
        )
        self.activation = activation

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.activation(x)
        return x


mlp_model = ConventionalMLP(layer_sizes=[1, 32, 32, 1])
print(mlp_model)
print(f"Trainable parameters: {sum(p.numel() for p in mlp_model.parameters())}")

#### Training

The model is trained with AdamW on random mini-batches of 128 points sampled uniformly from $[-2, 2]$ at every step. Mean squared error (MSE) is used as the loss function.

After 6 000 steps the loss typically plateaus at ~0.025 — a direct consequence of the difficulty MLPs have in representing the target.

In [ ]:
EPOCHS = 6000
mlp_train_losses = []
optimizer = torch.optim.AdamW(mlp_model.parameters(), lr=1e-2)

print("Starting training...")
for _ in range(EPOCHS):
    x_batch = rng.uniform(min_x, max_x, size=(128, 1))
    y_batch = func_1d(x_batch)
    x_batch = torch.tensor(x_batch, dtype=torch.float32)
    y_batch = torch.tensor(y_batch, dtype=torch.float32)

    optimizer.zero_grad()
    loss = torch.mean((mlp_model(x_batch) - y_batch) ** 2)
    loss.backward()
    optimizer.step()
    mlp_train_losses.append(loss.item())

print(f"Final training loss: {mlp_train_losses[-1]:.4f}")

## Fourier Network

A Fourier Network replaces the `QLinear + ReLU` building block with `QFourier` layers. Each `QFourier` layer computes a small learnable Fourier series for every input–output connection.

Because the nonlinearity is built into the layer definition (no separate activation function is needed), a Fourier Network with far fewer neurons can express the content better than an MLP with considerably more parameters. Crucially, the photonic NPU evaluates those cosines in a single optical pass rather than through iterative numerical approximation.

#### Initialization

A two-layer network with widths `[1, 4, 1]` and grid sizes `[4, 4]`. The grid size controls the number of frequency components $G$ per connection — larger grids capture finer detail at the cost of more parameters. Despite having roughly 16× fewer parameters than the MLP above, this compact architecture will achieve significantly lower loss.

In [ ]:
class FourierNet(torch.nn.Module):
    """Network of stacked QFourier layers — training runs on CPU, inference runs on NPU (eval mode)."""

    def __init__(self, layer_sizes, grid_sizes, add_bias=True):
        super().__init__()
        self.layers = torch.nn.ModuleList(
            [
                qant_layers.QFourier(
                    layer_sizes[i], layer_sizes[i + 1], grid_sizes[i], add_bias
                )
                for i in range(len(layer_sizes) - 1)
            ]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


fourier_model = FourierNet(layer_sizes=[1, 4, 1], grid_sizes=[4, 4])
print(fourier_model)
print(f"Trainable parameters: {sum(p.numel() for p in fourier_model.parameters())}")

#### Training

Identical training setup to the MLP: AdamW, MSE loss, 128-sample mini-batches, 6 000 steps. Note how the final training loss is roughly one order of magnitude lower than the MLP's despite the much smaller model — a clear demonstration of the Fourier representation advantage.

In [ ]:
EPOCHS = 6000
optimizer = torch.optim.AdamW(fourier_model.parameters(), lr=1e-2)
fourier_train_losses = []

print("Starting training...")
for _ in range(EPOCHS):
    x_batch = rng.uniform(min_x, max_x, size=(128, 1))
    y_batch = func_1d(x_batch)
    x_batch = torch.tensor(x_batch, dtype=torch.float32)
    y_batch = torch.tensor(y_batch, dtype=torch.float32)

    optimizer.zero_grad()
    loss = torch.mean((fourier_model(x_batch) - y_batch) ** 2)
    loss.backward()
    optimizer.step()
    fourier_train_losses.append(loss.item())

print(f"Final training loss: {fourier_train_losses[-1]:.4f}")

## Comparison: Fourier Network vs. Conventional MLP

We compare the two models along several axes:

| Metric | MLP | Fourier Network |
|---|---|---|
| Architecture | `[1, 32, 32, 1]` + ReLU | `[1, 4, 1]`, grid `[4, 4]` |
| Parameters | ~1 150 | ~70 |
| Training loss (6 000 steps) | ~0.025 | ~0.0036 |

The sections below examine parameter count, convergence speed, prediction accuracy, and raw inference throughput on the NPU.

#### Training Loss Curve

The log-scale loss curve shows two things at once: the Fourier Network converges faster and to a much lower final value. The second drop in the learning curve shows when the second frequency is learned.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    mlp_train_losses,
    label="MLP Training Loss",
)
plt.plot(fourier_train_losses, label="Fourier Network Training Loss")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.yscale("log")
plt.legend()
plt.show()

## Inference via the Q.ANT SDK

Switching the models to `eval()` mode activates the Q.ANT SDK routing in both `QLinear` and `QFourier` layers. From this point on, matrix multiplications and cosine evaluations are executed via Q.ANT SDK on the selected backend.

In [ ]:
nr_points = 200
x = np.linspace(min_x, max_x, nr_points)
x_tensor = torch.tensor(x[:, None], dtype=torch.float32)
y_true = func_1d(x)

# Switch to eval mode — QLinear and QFourier route through the NPU in eval mode
mlp_model.eval()
fourier_model.eval()

qant.generic.init_npu(0)
with torch.no_grad():
    # MLP
    y_pred_mlp = mlp_model(x_tensor).numpy().squeeze()

    # Fourier Network
    y_pred_fourier = fourier_model(x_tensor).numpy().squeeze()
qant.generic.release_npu(0)

#### Comparing Predictions Against Ground Truth

Both models are evaluated on 200 evenly-spaced points across $[-2, 2]$. The plot overlays ground truth (blue), MLP prediction (red), and Fourier Network prediction (green).

The Fourier Network typically tracks the high-frequency oscillations closely, while the MLP smooths over fine structure.

In [ ]:
print(
    "The Fourier Network learns periodic and high-frequency regions more effectively than MLP.\n"
)

test_loss_mlp = np.mean((y_true - y_pred_mlp) ** 2)
test_loss_fourier = np.mean((y_true - y_pred_fourier) ** 2)

fig = plt.figure(figsize=(8, 4))
plt.plot(x, y_true, label="Ground Truth", color="blue", linewidth=2, zorder=-2)
plt.plot(x, y_pred_mlp, label="MLP", color="red", linewidth=2)
plt.plot(x, y_pred_fourier, label="Fourier Network", color="green", linewidth=2)
plt.xlabel("x", fontsize=12)
plt.ylabel("y", fontsize=12)
plt.title(f"MLP Loss: {test_loss_mlp:.4f},  Fourier Loss: {test_loss_fourier:.4f}")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

## Challenge: Learning a Mixed-Frequency Function

The previous target function was a simple two-term sinusoid — a gentle warm-up. Now let's try something harder: a function that mixes a linear-frequency sinusoid with nonlinear-frequency components, creating structure that is difficult to represent with a fixed frequency basis:

$$g(x) = \sin(2\pi x) + \cos(\pi x^2) + \cos(\pi x^3)\,\sin(\pi x^3), \quad x \in [-2,\; 2]$$

Due to the quadratic and cubic phase, the local oscillation frequency increase with $|x|$. This produces faster oscillations toward the edges of the domain that are hard to capture with a small, fixed-frequency model.

**Your task:** design a `FourierNet` that fits $g(x)$ as well as possible.

In [ ]:
def func_hard(x: np.ndarray) -> np.ndarray:
    y = (
        np.sin(2 * np.pi * x)
        + np.cos(np.pi * x**2)
        + np.cos(np.pi * x**3) * np.sin(np.pi * x**3)
    )
    return y


min_x_hard = -2
max_x_hard = 2
x_hard = np.linspace(min_x_hard, max_x_hard, 1000)

plt.figure(figsize=(8, 4))
plt.plot(
    x_hard,
    func_hard(x_hard),
    label=r"$g(x) = \sin(2 \pi x) + \cos(\pi x^2) + \cos(\pi x^3) \sin(\pi x^3)$",
)
for kink in [-1, 0, 1]:
    plt.axvline(kink, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("x")
plt.ylabel("g(x)")
plt.title("Challenge target function (dashed lines mark non-differentiable kinks)")
plt.legend()
plt.grid(alpha=0.4)
plt.show()

#### Design Your Architecture

Use the `YourFourierNet` class to get the best possible fit.

Things to experiment with:
- Layer widths (`layer_sizes`) and depth
- Frequency components per connection (`grid_sizes`)
- Changes in the architecture of `YourFourierNet`

In [ ]:
class YourFourierNet(torch.nn.Module):
    """Network of stacked QFourier layers — training runs on CPU, inference runs on NPU (eval mode)."""

    def __init__(self, layer_sizes, grid_sizes, add_bias=True):
        super().__init__()
        self.layers = torch.nn.ModuleList(
            [
                qant_layers.QFourier(
                    layer_sizes[i], layer_sizes[i + 1], grid_sizes[i], add_bias
                )
                for i in range(len(layer_sizes) - 1)
            ]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x


my_fourier_model = YourFourierNet(
    layer_sizes=[1, 4, 1],  # try wider or deeper
    grid_sizes=[4, 4],
)

print(f"Trainable parameters: {sum(p.numel() for p in my_fourier_model.parameters())}")

### Training your Fourier model

In [ ]:
MY_EPOCHS = 8000
MY_LR = 5e-3

rng_hard = np.random.default_rng(seed=42)
optimizer_hard = torch.optim.AdamW(my_fourier_model.parameters(), lr=MY_LR)
hard_train_losses = []

print("Training...")
for _ in range(MY_EPOCHS):
    x_b = rng_hard.uniform(min_x_hard, max_x_hard, size=(128, 1))
    y_b = func_hard(x_b)
    x_b = torch.tensor(x_b, dtype=torch.float32)
    y_b = torch.tensor(y_b, dtype=torch.float32)

    optimizer_hard.zero_grad()
    loss = torch.mean((my_fourier_model(x_b) - y_b) ** 2)
    loss.backward()
    optimizer_hard.step()
    hard_train_losses.append(loss.item())

print(f"Final training loss: {hard_train_losses[-1]:.5f}")

### Evaluate your Fourier model

In [ ]:
x_eval = np.linspace(min_x_hard, max_x_hard, 500)
x_eval_tensor = torch.tensor(x_eval[:, None], dtype=torch.float32)
y_eval_true = func_hard(x_eval)

my_fourier_model.eval()
qant.generic.init_npu(0)
with torch.no_grad():
    y_eval_pred = my_fourier_model(x_eval_tensor).numpy().squeeze()
qant.generic.release_npu(0)

test_mse = np.mean((y_eval_true - y_eval_pred) ** 2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_eval, y_eval_true, color="blue", linewidth=2, label="Ground Truth")
axes[0].plot(
    x_eval,
    y_eval_pred,
    color="green",
    linewidth=2,
    linestyle="--",
    label="Your Fourier Network",
)
for kink in [-1, 0, 1]:
    axes[0].axvline(kink, color="gray", linestyle="--", alpha=0.4)
axes[0].set_title(f"Prediction vs. Ground Truth  (Test MSE: {test_mse:.5f})")
axes[0].set_xlabel("x")
axes[0].set_ylabel("g(x)")
axes[0].legend()
axes[0].grid(alpha=0.4)

axes[1].semilogy(hard_train_losses)
axes[1].set_title("Training Loss")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("MSE (log scale)")
axes[1].grid(alpha=0.4, which="both")

plt.tight_layout()
plt.show()

n_params = sum(p.numel() for p in my_fourier_model.parameters())
print(f"Trainable parameters : {n_params}")
print(f"Final train loss     : {hard_train_losses[-1]:.5f}")
print(f"Test MSE (on NPU)    : {test_mse:.5f}")